# License Plate Detection + OCR — Full Pipeline (Raspberry Pi code path)

Run this notebook cell-by-cell with the **License Plate Project (venv)** kernel (kernel picker, top-right in VS Code).

**This notebook runs the exact same code that runs on the Raspberry Pi** — the `license_plate_pipeline.pi` package: YOLO detection via **ONNX Runtime** and OCR via **RapidOCR** (both `onnxruntime`, no PyTorch / PaddlePaddle). So what you confirm here is what deploys. PaddlePaddle has no ARM64 wheel, which is why the Pi uses this ONNX/RapidOCR path — see `PROJECT_CONTEXT.md` and `IMPLEMENTATION_PLAN.md`.

**Steps:**
1. Setup
2. Load the ONNX detection model
3. Detect a plate in an image
4. Crop the detected plate
5. Load the OCR engine (RapidOCR)
6. Read the plate text
7. Final pipeline on one image: detect → read
8. Run the full pipeline on video — detection every frame, OCR only a few frames per **tracked** plate, then dedup. One clean row per truck.
9. Live webcam test — the same engine on a live feed

Full history, roadmap, and environment notes: [`PROJECT_CONTEXT.md`](../PROJECT_CONTEXT.md).

## 1. Setup

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))  # so `import license_plate_pipeline` works from notebooks/

TEST_IMAGES = PROJECT_ROOT / "test_images"
TEST_IMAGE = TEST_IMAGES / "demo2.jpg"  # US plate (California), ground truth 6FVZ747

## 2. Load the ONNX detection model

The YOLO26n model fine-tuned on Colab, exported to ONNX (`models/best.onnx`) and run with ONNX Runtime — the same file and runtime the Pi uses. No PyTorch / ultralytics needed to run inference.

In [ ]:
from license_plate_pipeline.pi.detection import get_model

session = get_model()  # onnxruntime InferenceSession from models/best.onnx
inp = session.get_inputs()[0]
print("ONNX detection model loaded. Input:", inp.name, inp.shape)

## 3. Detect a plate in an image

In [ ]:
from license_plate_pipeline.pi.detection import detect_boxes

img = cv2.imread(str(TEST_IMAGE))
boxes = detect_boxes(img)
print(f"Detections: {len(boxes)}")

annotated = img.copy()
for x1, y1, x2, y2 in boxes:
    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 3)

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO (ONNX) detection")
plt.show()

## 4. Crop the detected plate

`detect_boxes` already returns the box padded 10% per side. For RapidOCR we feed the crop as-is: the upscale + grayscale + equalize tuning that helped PaddleOCR gave RapidOCR no gain (grayscale actually hurt it), so `preprocess_for_ocr` in the Pi path is a pass-through. See `license_plate_pipeline/pi/ocr.py` for the measured reasoning.

In [ ]:
from license_plate_pipeline.pi.ocr import preprocess_for_ocr

x1, y1, x2, y2 = boxes[0]  # detect_boxes already padded this box
crop = img[y1:y2, x1:x2]
processed = preprocess_for_ocr(crop)  # identity for the RapidOCR path

plt.figure(figsize=(6, 3))
plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
plt.title("Plate crop fed to OCR")
plt.axis("off")
plt.show()

## 5. Load the OCR engine (RapidOCR)

RapidOCR runs the PP-OCR model family on ONNX Runtime — the same engine as the Pi, and it installs on ARM64 (PaddleOCR / PaddlePaddle can't). Runs on CPU here; actual Pi speed will differ.

In [ ]:
from license_plate_pipeline.pi.ocr import get_reader

reader = get_reader()  # RapidOCR (onnxruntime)
print("OCR engine loaded (RapidOCR).")

## 6. Read the plate text

In [ ]:
from license_plate_pipeline.pi.ocr import read_crop

for text, confidence in read_crop(crop):
    print(f"'{text}' (confidence: {confidence:.2f})")

## 7. Final pipeline on one image: detect → read

The single-image entry point from the Pi package (`license_plate_pipeline.pi.pipeline.read_plates_from_image`) — detect every plate, crop, OCR, return the plate-like text. Same detect + read used on video next.

In [ ]:
from license_plate_pipeline.pi.pipeline import read_plates_from_image

for text, confidence in read_plates_from_image(TEST_IMAGE):
    print(f"Detected plate text: '{text}' (confidence: {confidence:.2f})")

## 8. Run the full pipeline on video

`process_video` (from `license_plate_pipeline.pi.pipeline`) is what the Pi uses to log trucks. Each frame it runs cheap YOLO detection, **tracks** every plate's box across frames (IOU matching), and runs the expensive OCR only a few times per *physical* plate — not every frame. That single change took this clip from ~1170s to ~100s. It then dedups (merges OCR flicker, absorbs partial reads) and returns one clean row per truck: plate text, first/last seen, frame count, best confidence.

`demo.mp4` is ~21s; expect roughly a minute or two on this CPU. (The tracking + dedup that used to be shown as separate manual steps now all live inside this one call — see `license_plate_pipeline/tracking.py` and `dedup.py`.)

In [ ]:
from license_plate_pipeline.pi.pipeline import process_video

video_path = TEST_IMAGES / "demo.mp4"
events = process_video(video_path)
print(f"Final deduped events: {len(events)}")

In [ ]:
import pandas as pd

df = pd.DataFrame(events).sort_values("first_seen").reset_index(drop=True)
df["seconds_first"] = df["first_seen"].round(2)
df["seconds_last"] = df["last_seen"].round(2)
df[["plate_text", "seconds_first", "seconds_last", "frame_count", "best_confidence"]]

In [ ]:
# Verify the tracking/dedup decisions: which raw OCR reads merged into each event.
for e in sorted(events, key=lambda ev: ev["first_seen"]):
    reads = ", ".join(f"'{t}' ({c:.2f})" for t, c, _fc in e["readings"])
    print(f"-> '{e['plate_text']}' (best {e['best_confidence']:.2f}): {reads}")

## 9. Live webcam test

Same Pi engine (ONNX + RapidOCR), live on a webcam instead of a file — this mirrors `pi_scripts/webcam_test.py`, the sanity check you'll run on the Pi itself. It OCRs every frame (no tracking), which is fine for a quick "does it detect + read live" check; the truck-logging deployment uses the tracked `process_video` from step 8.

**Run the next cell yourself, interactively** — it opens a live window and needs you to hold a plate / play a recording in front of the camera. I can't watch a webcam feed, so I can't run this one for you.

- A window titled "Live webcam" shows the feed with detections boxed
- Detected readings print below the cell with a timestamp
- **Press `q` in the video window to stop**

In [ ]:
import time

from license_plate_pipeline.pi.detection import detect_boxes
from license_plate_pipeline.pi.ocr import read_crop, select_plate_text

CAMERA_INDEX = 1  # external camera - 0 is usually the built-in laptop webcam

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)
if not cap.isOpened():
    raise RuntimeError(f"Could not open camera (index {CAMERA_INDEX}) - check it's connected and not in use by another app")

# Lower capture resolution - smaller frames mean faster detection + smaller OCR crops.
# Actual resolution may differ slightly (driver picks the nearest supported mode).
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 480)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 320)
print(f"Capture resolution: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")
print("Webcam open. Press 'q' in the video window to stop.")

live_readings = []
start_time = time.time()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            print("Failed to read a frame from the webcam - stopping.")
            break

        for x1, y1, x2, y2 in detect_boxes(frame):
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            crop = frame[y1:y2, x1:x2]
            picked = select_plate_text(read_crop(crop))
            if picked:
                text, confidence = picked
                elapsed = time.time() - start_time
                print(f"[{elapsed:6.2f}s] '{text}' (confidence: {confidence:.2f})")
                live_readings.append({"plate_text": text, "confidence": confidence, "seconds": elapsed})
                cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        cv2.imshow("Live webcam - press 'q' to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()

print(f"\nStopped. Captured {len(live_readings)} raw readings.")

In [41]:
if live_readings:
    pd.DataFrame(live_readings)
else:
    print("No readings captured - run the cell above first.")

## What's next

This notebook now runs the exact Pi code path end to end. Once the live webcam test (step 9) reads plates cleanly, the next step is running it on the actual Raspberry Pi — follow [`PI_SETUP.md`](../PI_SETUP.md). Project status and roadmap live in [`PROJECT_CONTEXT.md`](../PROJECT_CONTEXT.md).